In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from langchain_chroma import Chroma


import numpy as np
from typing import List
from euriai import *


C:\Users\DELL\AppData\Local\Temp\ipykernel_4872\1594028491.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [3]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [4]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i , doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : C:\Users\DELL\AppData\Local\Temp\tmpzacuxfhk


In [5]:
from langchain_community.document_loaders import DirectoryLoader
import os

loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}

)

doc = loader.load()


if not doc:
    print(f"WARNING: No documents found in '{temp_dir}' matching glob '*.txt'.")
else:
    print(f"Loaded {len(doc)} documents")
    print(f"\nFirst document preview:")
    print(doc[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [6]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

In [7]:
from posixpath import sep

from transformers.masking_utils import chunked_overlay


text_split =RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function =len,
    separators=[" "]
)

chunks = text_split.split_documents(doc)


print(f"Created {len(chunks)} chunks from {len(doc)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_0.txt'}


In [8]:
os.environ["OPENAI_API_KEY"] = os.getenv("EURI_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://api.euron.one/api/v1/euri"


In [9]:
sample_text="MAchine LEarning is fascinating"
embeddings=OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001F180AD7450>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001F180AE9510>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [10]:
vector = embeddings.embed_query(sample_text)
vector

[-0.022837752476334572,
 0.011149656027555466,
 0.00850998517125845,
 -0.029968805611133575,
 -0.005709438119083643,
 0.020487001165747643,
 -0.003222434548661113,
 0.011412310414016247,
 -0.021445687860250473,
 -0.04412584751844406,
 0.006356223486363888,
 0.04383692890405655,
 -0.018267575651407242,
 0.005177564453333616,
 -0.0012697672937065363,
 0.003861011704429984,
 0.03685033693909645,
 0.009560600854456425,
 0.004277974832803011,
 -0.007689191959798336,
 -0.02157701551914215,
 0.020959777757525444,
 -0.005358138587325811,
 -0.040553756058216095,
 -0.007387139834463596,
 0.012423527427017689,
 0.01457728911191225,
 -0.036456357687711716,
 -0.013723663985729218,
 -0.0023523936979472637,
 0.016809847205877304,
 -0.008890833705663681,
 -0.01929192617535591,
 -0.040790144354104996,
 -0.015115729533135891,
 -0.012134608812630177,
 0.00017698356532491744,
 0.0012303692055866122,
 -0.014853076077997684,
 -0.0009463747264817357,
 0.017295757308602333,
 0.012784676626324654,
 -0.01059808

In [11]:
chunks

[Document(metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset

In [12]:
persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store named : {vectorstore._collection.name} ")
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store named : rag_collection 
Vector store created with 15 vectors
Persisted to: ./chroma_db


In [13]:
query="What are the types of machine learning?"

similar = vectorstore.similarity_search(query,k=3)

In [14]:
similar

[Document(id='903f86b6-c1e0-4377-a6f9-bce3011720c0', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(id='23e12907-45b8-49d7-8fcd-c1b308baa00d', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are 

In [15]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(id='6ed8723c-6a23-48bb-b7c5-a8857dc2ded4', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(id='9132c8c0-e1b0-42c7-9ecf-ad19c5b749fc', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentimen

In [16]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(id='6ed8723c-6a23-48bb-b7c5-a8857dc2ded4', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.2334703505039215),
 (Document(id='9132c8c0-e1b0-42c7-9ecf-ad19c5b749fc', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named ent

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


In [22]:
test_response=llm.invoke("What is Large Language Models")
test_response

AIMessage(content='Large Language Models (LLMs) are advanced artificial intelligence systems designed to understand, generate, and manipulate human language. They are built using deep learning techniques, particularly neural networks with many layers, and are trained on vast amounts of text data from books, websites, articles, and other written sources.\n\n**Key features of Large Language Models include:**\n\n1. **Scale:** LLMs contain billions or even trillions of parameters (the internal settings of the model that are learned during training). The large number of parameters allows them to capture complex language patterns.\n\n2. **Training Data:** They are trained on diverse and extensive datasets to learn grammar, facts, reasoning abilities, and even some level of common sense.\n\n3. **Capabilities:** LLMs can perform a wide range of language-related tasks, such as:\n   - Text completion and generation\n   - Translation between languages\n   - Summarizing documents\n   - Answering q

In [23]:
from langchain.chat_models.base import init_chat_model

llm=init_chat_model("openai:gpt-4.1-mini") 
#llm=init_chat_model("groq:")
llm

ChatOpenAI(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}}, output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F1868EC0D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F1868EC710>, root_client=<openai.OpenAI object at 0x000001F18688C2D0>, root_async_client=<openai.AsyncOpenA

In [24]:
import langchain
print(langchain.__version__)

1.3.7


In [25]:
from langchain_core.prompts import ChatPromptTemplate


custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}
                                                 
Question: {question}

Answer: """)

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: "), additional_kwargs={})])

In [26]:
# make retriver

retriever = vectorstore.as_retriever(
    search_kwarg = {"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F181AC6710>, search_kwargs={})

In [39]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs )

In [ ]:

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


rag_chain_lcel = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }

    | custom_prompt
    | llm
    |StrOutputParser()
)

In [45]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep learning has revolutionized fields like computer vision, natural language processing, and speech recognition.'

In [50]:
retriever.invoke("what is learning")

[Document(id='903f86b6-c1e0-4377-a6f9-bce3011720c0', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(id='23e12907-45b8-49d7-8fcd-c1b308baa00d', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are 

In [51]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [52]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning are learning through interaction with an environment using rewards and penalties. This means that reinforcement learning involves an agent that takes actions in an environment and learns to improve its behavior based on the feedback it receives in the form of rewards (positive feedback) or penalties (negative feedback). This process enables the system to learn from experience without explicit programming.

Source Documents:

--- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 3 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penaltie

In [53]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [55]:
retriever = vectorstore.as_retriever(
    search_kwarg = {"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F181AC6710>, search_kwargs={})

In [58]:
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""


prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")]
)

In [59]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [60]:
document_chain = create_stuff_documents_chain(llm,prompt)

rag_chain = create_retrieval_chain(retriever,document_chain)

In [61]:
response=rag_chain.invoke({"input":"What is Deep LEarning"})

In [62]:
response

{'input': 'What is Deep LEarning',
 'context': [Document(id='9206198a-07c1-441a-9556-f037b2e48db0', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(id='46f9dc7c-487d-47c4-8f2e-33d4f786050b', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks

In [ ]:
# # 1. Create a prompt template and document chain
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "Answer the user question based on context: {context}"),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}")
# ])


# llm = init_chat_model("openai:gpt-4.1-mini")
# doc_chain = create_stuff_documents_chain(llm, prompt)


# # 2. Build history-aware retriever & retrieval chain
# history_retriever = create_history_aware_retriever(llm, retriever, prompt)
# rag_chain = create_retrieval_chain(history_retriever, doc_chain)


# # 3. Invoke with history
# response = rag_chain.invoke({
#     "input": "Tell me more about it", 
#     "chat_history": [
#         HumanMessage(content="What is RAG?"), 
#         AIMessage(content="RAG is Retrieval-Augmented Generation...")
#     ]
# })

In [63]:
vectorstore

In [64]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [65]:
chunks

[Document(metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpzacuxfhk\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset

In [67]:
new_doc = Document(
    page_content=new_document,
    metadata= {"source":"manual_addition","topic":"reinforcement_learning"}
)

In [68]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [70]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents([new_doc])


In [71]:
chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [72]:
vectorstore.add_documents(chunks)

['ed7d64f9-1468-4f3d-81df-cbdf4cbe9cbb',
 'fb871ddf-fa2b-4687-a65f-073bafc71d75']

In [75]:
print(len(chunks))
vectorstore._collection.count()

2


17

In [76]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [79]:
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")

])

In [90]:

history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_prompt)


history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F181AC6710>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChun

In [83]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""


qa_prompt = ChatPromptTemplate.from_messages([
    ("system",qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

In [85]:
question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [87]:
chat_history = []

result1 = conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":" What is machine learning"
})

print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, unsupervised learning, and reinforcement learning. These methods allow models to train on data, find patterns, or learn through interaction.


In [88]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [89]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, unsupervised learning, and reinforcement learning. These methods allow models to train on data, find patterns, or learn through interaction.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(id='903f86b6-c1e0-4377-a6f9-bce3011720c0', metadata={'source': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmpprgvp1ov\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machi